## Ontology Builder

In [8]:
!pip install rdflib

## General Ontology

In [ ]:
from rdflib import Graph, Literal, Namespace, RDF, RDFS
from rdflib.namespace import OWL, XSD
import re

BMO = Namespace("http://www.bridge-management-ontology.org/bmo#")
g = Graph()
g.bind("bmo", BMO)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

top_level_classes = {
    "Asset": "A physical entity that is owned and has value, such as a bridge or its components.",
    "Event": "An occurrence that happens at a specific point in time related to an asset.",
    "ExternalFactor": "An external condition or influence that affects an asset."
}
for name, comment in top_level_classes.items():
    class_uri = BMO[name]
    g.add((class_uri, RDF.type, OWL.Class))
    g.add((class_uri, RDFS.label, Literal(name)))
    g.add((class_uri, RDFS.comment, Literal(comment)))


bridge_class = BMO.Bridge
g.add((bridge_class, RDF.type, OWL.Class))
g.add((bridge_class, RDFS.subClassOf, BMO.Asset))
g.add((bridge_class, RDFS.label, Literal("Bridge")))

bridge_element_class = BMO.BridgeElement
g.add((bridge_element_class, RDF.type, OWL.Class))
g.add((bridge_element_class, RDFS.subClassOf, BMO.Asset))
g.add((bridge_element_class, RDFS.label, Literal("Bridge Element")))

inspection_event_class = BMO.InspectionEvent
g.add((inspection_event_class, RDF.type, OWL.Class))
g.add((inspection_event_class, RDFS.subClassOf, BMO.Event))
g.add((inspection_event_class, RDFS.label, Literal("Inspection Event")))

maintenance_event_class = BMO.MaintenanceEvent
g.add((maintenance_event_class, RDF.type, OWL.Class))
g.add((maintenance_event_class, RDFS.subClassOf, BMO.Event))
g.add((maintenance_event_class, RDFS.label, Literal("Maintenance Event")))

traffic_profile_class = BMO.TrafficProfile
g.add((traffic_profile_class, RDF.type, OWL.Class))
g.add((traffic_profile_class, RDFS.subClassOf, BMO.ExternalFactor))
g.add((traffic_profile_class, RDFS.label, Literal("Traffic Profile")))

environmental_exposure_class = BMO.EnvironmentalExposure
g.add((environmental_exposure_class, RDF.type, OWL.Class))
g.add((environmental_exposure_class, RDFS.subClassOf, BMO.ExternalFactor))
g.add((environmental_exposure_class, RDFS.label, Literal("Environmental Exposure")))



# A Bridge has Bridge Elements
g.add((BMO.hasElement, RDF.type, OWL.ObjectProperty))
g.add((BMO.hasElement, RDFS.domain, bridge_class))
g.add((BMO.hasElement, RDFS.range, bridge_element_class))

# An Asset undergoes an Event
g.add((BMO.undergoesEvent, RDF.type, OWL.ObjectProperty))
g.add((BMO.undergoesEvent, RDFS.domain, BMO.Asset))
g.add((BMO.undergoesEvent, RDFS.range, BMO.Event))

# An Asset is influenced by an External Factor
g.add((BMO.isInfluencedBy, RDF.type, OWL.ObjectProperty))
g.add((BMO.isInfluencedBy, RDFS.domain, BMO.Asset))
g.add((BMO.isInfluencedBy, RDFS.range, BMO.ExternalFactor))



def create_data_property(column_name, domain_class, data_type=XSD.string):
    """Helper function to create a data property and link it to its domain class."""
    prop_name_camel = ''.join(x.capitalize() for x in re.split(r'[\s\(\).\-/%;?]', column_name) if x)
    prop_name_camel = prop_name_camel[0].lower() + prop_name_camel[1:] if prop_name_camel else column_name.lower()
    
    prop_uri = BMO[prop_name_camel]
    g.add((prop_uri, RDF.type, OWL.DatatypeProperty))
    g.add((prop_uri, RDFS.label, Literal(column_name)))
    g.add((prop_uri, RDFS.domain, domain_class))
    g.add((prop_uri, RDFS.range, data_type))

# --- Asset -> Bridge Properties (Structural, Location, Admin) ---
create_data_property("STRUCTURE_NUMBER", bridge_class, XSD.string)
create_data_property("NBI_27 - Year Built", bridge_class, XSD.gYear)
create_data_property("NBI_106 - Year Reconstructed", bridge_class, XSD.gYear)
create_data_property("NBI_Bridge Age (yr)", bridge_class, XSD.integer)

# Structural Properties
create_data_property("NBI_43A - Main Span Material", bridge_class, XSD.string)
create_data_property("NBI_44A - Approach Spans Material", bridge_class, XSD.string)
create_data_property("NBI_46 - Number of Approach Spans", bridge_class, XSD.integer)
create_data_property("NBI_48 - Length of Maximum Span (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_49 - Structure Length (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_112 - NBIS Bridge Length", bridge_class, XSD.string) # NBIS length can be a flag
create_data_property("NBI_51 - Bridge Roadway Width - Curb to Curb (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_52 - Deck Width - Out to Out (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_CAT29 - Deck Area (sq. ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_107 - Deck Structure Type Code", bridge_class, XSD.string)
create_data_property("NBI_108A - Wearing Surface Type Code", bridge_class, XSD.string)
create_data_property("NBI_108B - Membrane Type Code", bridge_class, XSD.string)
create_data_property("NBI_31 - Design Load", bridge_class, XSD.string)
create_data_property("NBI_32 - Approach Roadway Width (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_10 - Inventory Route, Minimum Vertical Clearance (ft.)", bridge_class, XSD.decimal)

# Location & Admin Properties
create_data_property("OWNER_AGENCY", bridge_class, XSD.string)
create_data_property("NBI_1 - State Code", bridge_class, XSD.string)
create_data_property("NBI_3 - County Code", bridge_class, XSD.string)
create_data_property("NBI_4 - Place Code", bridge_class, XSD.string)
create_data_property("NBI_9 - Administrative Area", bridge_class, XSD.string)
create_data_property("NBI_21 - Maintenance Responsibility", bridge_class, XSD.string)
create_data_property("NBI_26 - Functional Classification of Inventory Route", bridge_class, XSD.string)
create_data_property("NBI_104 - Highway System of Inventory Route", bridge_class, XSD.string)
create_data_property("NBI_37 - Historical Significance Code", bridge_class, XSD.string)


# --- Asset -> BridgeElement Properties ---
create_data_property("ELEMENT_CATEGORY", bridge_element_class, XSD.string)
create_data_property("PARENT_ELEMENT_NUMBER", bridge_element_class, XSD.string)
create_data_property("ELEMENT_NAME", bridge_element_class, XSD.string)
create_data_property("ELEMENT_UNIT", bridge_element_class, XSD.string)
create_data_property("TOTAL_QUANTITY", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_1", bridge_element_class, XSD.decimal) 
create_data_property("CS_PERCENT_1", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_2", bridge_element_class, XSD.decimal)
create_data_property("CS_PERCENT_2", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_3", bridge_element_class, XSD.decimal)
create_data_property("CS_PERCENT_3", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_4", bridge_element_class, XSD.decimal)
create_data_property("CS_PERCENT_4", bridge_element_class, XSD.decimal)
create_data_property("HI", bridge_element_class, XSD.decimal) 

# --- Event -> InspectionEvent Properties ---
create_data_property("NBI_90 - Inspection Date", inspection_event_class, XSD.date)
create_data_property("NBI_91 - Designated Inspection Frequency", inspection_event_class, XSD.integer)
create_data_property("NBI_CAT10 - Bridge Condition", inspection_event_class, XSD.string)
create_data_property("NBI_92A - Fracture Critical Details", inspection_event_class, XSD.string)
create_data_property("NBI_92B - Underwater Inspection", inspection_event_class, XSD.string)
create_data_property("NBI_92C - Other Special Inspection", inspection_event_class, XSD.string)

# --- Event -> MaintenanceEvent Properties ---
create_data_property("NBI_94 - Bridge Improvement Cost", maintenance_event_class, XSD.decimal)
create_data_property("NBI_95 - Roadway Improvement Cost", maintenance_event_class, XSD.decimal)
create_data_property("NBI_76 - Length of Structure Improvement (ft.)", maintenance_event_class, XSD.decimal)

# --- ExternalFactor -> TrafficProfile Properties ---
create_data_property("NBI_29 - Average Daily Traffic", traffic_profile_class, XSD.integer)
create_data_property("NBI_109 - Average Daily Truck Traffic (%)", traffic_profile_class, XSD.decimal) # Changed to decimal
create_data_property("NBI_30 - Year of Average Daily Traffic", traffic_profile_class, XSD.gYear)

# --- ExternalFactor -> EnvironmentalExposure Properties ---
create_data_property("NBI_Avg Daily Max Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Daily Min Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Daily Wind Speed", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Monthly Max Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Monthly Min Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Freeze Thaw Cycles", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Number of Days with Measurable Precipitation", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Number of Days with Temperature Below 0?C", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Number of Snowfall Days", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Prevailing Wind Direction", environmental_exposure_class, XSD.string)
create_data_property("NBI_Time of Wetness", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Total Precipitation", environmental_exposure_class, XSD.decimal)


try:
    g.serialize(destination="BridgeOntology1022.owl", format="pretty-xml", encoding="utf-8")
except Exception as e:
    print(f"An error occurred while saving the file: {e}")



## Newer version w/ neo4j compatability

In [1]:
!pip install neo4j rdflib-neo4j

In [2]:
import re
from rdflib import Graph, Literal, Namespace, RDF, RDFS
from rdflib.namespace import OWL, XSD
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY
from neo4j import GraphDatabase

# --- 1. SET UP NEO4J CONNECTION DETAILS ---
uri = "bolt://localhost:7687"
user = "neo4j"
password = "MikalBridges"
database = "neo4j"

# --- 2. NATIVE NEO4J CONNECTION TO CLEAR DB ---
print("Connecting directly to Neo4j to clear database...")
driver = None
try:
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session(database=database) as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("Database cleared successfully.")
except Exception as e:
    print(f"\n--- NATIVE NEO4J ERROR ---")
    print(f"Could not clear database: {e}")
    exit()
finally:
    if driver:
        driver.close()

# --- 3. RDFLIB-NEO4J CONNECTION TO LOAD SCHEMA ---
print(f"Connecting to Neo4j at {uri} with rdflib...")
try:
    auth_data = {
        'uri': uri,
        'user': user,
        'pwd': password,
        'database': database
    }

    config = Neo4jStoreConfig(
        auth_data=auth_data,
        handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
        batching=True
    )

    g = Graph(store=Neo4jStore(config=config))
    
    print("rdflib connection successful. Setting up namespaces...")
    
    # --- 4. SET UP GRAPH AND NAMESPACES ---
    BMO = Namespace("http://www.bridge-management-ontology.org/bmo#")
    g.bind("bmo", BMO)
    g.bind("owl", OWL)
    g.bind("rdfs", RDFS)
    g.bind("xsd", XSD)
    
    # --- 3. HELPER FUNCTION ---
    def create_data_property(column_name, domain_class, data_type=XSD.string):
        prop_name = re.sub(r'[^\w\s-]', '', column_name)
        prop_name_parts = re.split(r'[\s\(\).\-/]', prop_name)
        prop_name_camel = ''.join(x.capitalize() for x in prop_name_parts if x)
        
        if not prop_name_camel:
            return
    
        prop_name_camel = prop_name_camel[0].lower() + prop_name_camel[1:]
        
        prop_uri = BMO[prop_name_camel]
        g.add((prop_uri, RDF.type, OWL.DatatypeProperty))
        g.add((prop_uri, RDFS.label, Literal(column_name)))
        g.add((prop_uri, RDFS.domain, domain_class))
        g.add((prop_uri, RDFS.range, data_type))
    
    # --- 4. DEFINE CLASSES ---
    print("Defining classes and writing to Neo4j...")
    
    # Top-level classes
    asset_class = BMO.Asset
    g.add((asset_class, RDF.type, OWL.Class))
    g.add((asset_class, RDFS.label, Literal("Asset")))
    
    event_class = BMO.Event
    g.add((event_class, RDF.type, OWL.Class))
    g.add((event_class, RDFS.label, Literal("Event")))
    
    # Asset Layer
    bridge_class = BMO.Bridge
    g.add((bridge_class, RDF.type, OWL.Class))
    g.add((bridge_class, RDFS.subClassOf, asset_class))
    g.add((bridge_class, RDFS.label, Literal("Bridge")))
    
    bridge_element_class = BMO.BridgeElement
    g.add((bridge_element_class, RDF.type, OWL.Class))
    g.add((bridge_element_class, RDFS.subClassOf, asset_class))
    g.add((bridge_element_class, RDFS.label, Literal("Bridge Element")))
    
    # Environment & Exposure Layer
    location_class = BMO.Location
    g.add((location_class, RDF.type, OWL.Class))
    g.add((location_class, RDFS.label, Literal("Location")))
    
    weather_exposure_class = BMO.WeatherExposure
    g.add((weather_exposure_class, RDF.type, OWL.Class))
    g.add((weather_exposure_class, RDFS.label, Literal("Weather Exposure")))
    
    # *** NEW: ClimateZone class for GNN ***
    climate_zone_class = BMO.ClimateZone
    g.add((climate_zone_class, RDF.type, OWL.Class))
    g.add((climate_zone_class, RDFS.label, Literal("Climate Zone")))
    g.add((climate_zone_class, RDFS.comment, Literal("Regional climate classification based on freeze-thaw severity")))
    
    traffic_profile_class = BMO.TrafficProfile
    g.add((traffic_profile_class, RDF.type, OWL.Class))
    g.add((traffic_profile_class, RDFS.label, Literal("Traffic Profile")))
    
    # *** NEW: TrafficPattern class for GNN ***
    traffic_pattern_class = BMO.TrafficPattern
    g.add((traffic_pattern_class, RDF.type, OWL.Class))
    g.add((traffic_pattern_class, RDFS.label, Literal("Traffic Pattern")))
    g.add((traffic_pattern_class, RDFS.comment, Literal("Traffic intensity classification (LOW/MEDIUM/HIGH)")))
    
    # *** NEW: Material class for GNN ***
    material_class = BMO.Material
    g.add((material_class, RDF.type, OWL.Class))
    g.add((material_class, RDFS.label, Literal("Material")))
    g.add((material_class, RDFS.comment, Literal("Construction material type")))
    
    # Deterioration & Maintenance Layer
    maintenance_activity_class = BMO.MaintenanceActivity
    g.add((maintenance_activity_class, RDF.type, OWL.Class))
    g.add((maintenance_activity_class, RDFS.subClassOf, event_class))
    g.add((maintenance_activity_class, RDFS.label, Literal("Maintenance Activity")))
    
    inspection_class = BMO.Inspection
    g.add((inspection_class, RDF.type, OWL.Class))
    g.add((inspection_class, RDFS.subClassOf, maintenance_activity_class))
    g.add((inspection_class, RDFS.label, Literal("Inspection")))
    
    maintenance_action_class = BMO.MaintenanceAction
    g.add((maintenance_action_class, RDF.type, OWL.Class))
    g.add((maintenance_action_class, RDFS.subClassOf, maintenance_activity_class))
    g.add((maintenance_action_class, RDFS.label, Literal("Maintenance Action")))
    
    deterioration_process_class = BMO.DeteriorationProcess
    g.add((deterioration_process_class, RDF.type, OWL.Class))
    g.add((deterioration_process_class, RDFS.label, Literal("Deterioration Process")))
    
    condition_rating_class = BMO.ConditionRating
    g.add((condition_rating_class, RDF.type, OWL.Class))
    g.add((condition_rating_class, RDFS.label, Literal("Condition Rating")))
    
    deterioration_prediction_class = BMO.DeteriorationPrediction
    g.add((deterioration_prediction_class, RDF.type, OWL.Class))
    g.add((deterioration_prediction_class, RDFS.label, Literal("Deterioration Prediction")))
    
    # --- 5. DEFINE OBJECT PROPERTIES (Relationships) ---
    print("Defining object properties and writing to Neo4j...")
    
    # Asset Layer
    has_element_prop = BMO.hasElement
    is_part_of_prop = BMO.isPartOf
    g.add((has_element_prop, RDF.type, OWL.ObjectProperty))
    g.add((has_element_prop, RDFS.domain, bridge_class))
    g.add((has_element_prop, RDFS.range, bridge_element_class))
    g.add((is_part_of_prop, RDF.type, OWL.ObjectProperty))
    g.add((is_part_of_prop, RDFS.domain, bridge_element_class))
    g.add((is_part_of_prop, RDFS.range, bridge_class))
    
    # Environment Layer
    g.add((BMO.locatedIn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.locatedIn, RDFS.domain, bridge_class))
    g.add((BMO.locatedIn, RDFS.range, location_class))
    
    g.add((BMO.hasExposure, RDF.type, OWL.ObjectProperty))
    g.add((BMO.hasExposure, RDFS.domain, location_class))
    g.add((BMO.hasExposure, RDFS.range, weather_exposure_class))
    
    # *** NEW: EXPOSED_TO relationship for GNN ***
    g.add((BMO.exposedTo, RDF.type, OWL.ObjectProperty))
    g.add((BMO.exposedTo, RDFS.domain, bridge_class))
    g.add((BMO.exposedTo, RDFS.range, climate_zone_class))
    g.add((BMO.exposedTo, RDFS.label, Literal("exposed to")))
    
    g.add((BMO.hasTrafficProfile, RDF.type, OWL.ObjectProperty))
    g.add((BMO.hasTrafficProfile, RDFS.domain, bridge_class))
    g.add((BMO.hasTrafficProfile, RDFS.range, traffic_profile_class))
    
    # *** NEW: HAS_TRAFFIC relationship for GNN ***
    g.add((BMO.hasTraffic, RDF.type, OWL.ObjectProperty))
    g.add((BMO.hasTraffic, RDFS.domain, bridge_class))
    g.add((BMO.hasTraffic, RDFS.range, traffic_pattern_class))
    g.add((BMO.hasTraffic, RDFS.label, Literal("has traffic")))
    
    # *** NEW: CONSTRUCTED_WITH relationship for GNN ***
    g.add((BMO.constructedWith, RDF.type, OWL.ObjectProperty))
    g.add((BMO.constructedWith, RDFS.domain, bridge_class))
    g.add((BMO.constructedWith, RDFS.range, material_class))
    g.add((BMO.constructedWith, RDFS.label, Literal("constructed with")))
    
    # *** NEW: Spatial relationships for GNN ***
    g.add((BMO.near, RDF.type, OWL.ObjectProperty))
    g.add((BMO.near, RDFS.domain, bridge_class))
    g.add((BMO.near, RDFS.range, bridge_class))
    g.add((BMO.near, RDFS.label, Literal("near")))
    g.add((BMO.near, RDFS.comment, Literal("Spatial proximity relationship")))
    
    g.add((BMO.similarTo, RDF.type, OWL.ObjectProperty))
    g.add((BMO.similarTo, RDFS.domain, bridge_class))
    g.add((BMO.similarTo, RDFS.range, bridge_class))
    g.add((BMO.similarTo, RDFS.label, Literal("similar to")))
    g.add((BMO.similarTo, RDFS.comment, Literal("Similarity based on material, age, traffic")))
    
    g.add((BMO.trafficInducesStressOn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.trafficInducesStressOn, RDFS.domain, traffic_profile_class))
    g.add((BMO.trafficInducesStressOn, RDFS.range, bridge_class))
    
    g.add((BMO.weatherInducesStressOn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.weatherInducesStressOn, RDFS.domain, weather_exposure_class))
    g.add((BMO.weatherInducesStressOn, RDFS.range, bridge_class))
    
    # Deterioration & Maintenance Layer
    g.add((BMO.experiences, RDF.type, OWL.ObjectProperty))
    g.add((BMO.experiences, RDFS.domain, asset_class))
    g.add((BMO.experiences, RDFS.range, deterioration_process_class))
    
    g.add((BMO.observedBy, RDF.type, OWL.ObjectProperty))
    g.add((BMO.observedBy, RDFS.domain, deterioration_process_class))
    g.add((BMO.observedBy, RDFS.range, inspection_class))
    
    g.add((BMO.resultedIn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.resultedIn, RDFS.domain, inspection_class))
    g.add((BMO.resultedIn, RDFS.range, condition_rating_class))
    
    g.add((BMO.triggers, RDF.type, OWL.ObjectProperty))
    g.add((BMO.triggers, RDFS.domain, condition_rating_class))
    g.add((BMO.triggers, RDFS.range, maintenance_action_class))
    
    g.add((BMO.isPerformedOn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.isPerformedOn, RDFS.domain, maintenance_activity_class))
    g.add((BMO.isPerformedOn, RDFS.range, asset_class))
    
    # Prediction Layer
    g.add((BMO.predictionTriggersAction, RDF.type, OWL.ObjectProperty))
    g.add((BMO.predictionTriggersAction, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.predictionTriggersAction, RDFS.range, maintenance_action_class))
    
    g.add((BMO.derivedFromWeather, RDF.type, OWL.ObjectProperty))
    g.add((BMO.derivedFromWeather, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.derivedFromWeather, RDFS.range, weather_exposure_class))
    
    g.add((BMO.derivedFromTraffic, RDF.type, OWL.ObjectProperty))
    g.add((BMO.derivedFromTraffic, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.derivedFromTraffic, RDFS.range, traffic_profile_class))
    
    g.add((BMO.predictsFor, RDF.type, OWL.ObjectProperty))
    g.add((BMO.predictsFor, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.predictsFor, RDFS.range, asset_class))
    
    # --- 6. DEFINE DATA PROPERTIES ---
    print("Defining data properties and writing to Neo4j...")
    
    # Bridge (existing properties)
    create_data_property("STRUCTURE_NUMBER", bridge_class, XSD.string)
    create_data_property("LATEST_STRUCTURE_NUMBER", bridge_class, XSD.string)
    create_data_property("NBI_27 - Year Built", bridge_class, XSD.gYear)
    create_data_property("NBI_106 - Year Reconstructed", bridge_class, XSD.gYear)
    create_data_property("NBI_Bridge Age (yr)", bridge_class, XSD.integer)
    create_data_property("NBI_43A - Main Span Material", bridge_class, XSD.string)
    create_data_property("NBI_43B - Main Span Design", bridge_class, XSD.string)
    create_data_property("NBI_49 - Structure Length (ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_52 - Deck Width - Out to Out (ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_29 - Average Daily Traffic", bridge_class, XSD.integer)
    create_data_property("OWNER_AGENCY", bridge_class, XSD.string)
    
    # *** NEW: Computed properties for GNN ***
    create_data_property("adtl", bridge_class, XSD.decimal)  # Traffic per lane
    create_data_property("truckPercentage", bridge_class, XSD.decimal)
    create_data_property("currentConditionRating", bridge_class, XSD.integer)
    create_data_property("criticalityScore", bridge_class, XSD.decimal)
    
    # ClimateZone properties
    create_data_property("zoneID", climate_zone_class, XSD.string)
    create_data_property("severity", climate_zone_class, XSD.string)
    create_data_property("freezeThawMin", climate_zone_class, XSD.integer)
    create_data_property("freezeThawMax", climate_zone_class, XSD.integer)
    
    # TrafficPattern properties
    create_data_property("patternID", traffic_pattern_class, XSD.string)
    create_data_property("intensity", traffic_pattern_class, XSD.string)
    create_data_property("adtlMin", traffic_pattern_class, XSD.integer)
    create_data_property("adtlMax", traffic_pattern_class, XSD.integer)
    
    # Material properties
    create_data_property("materialType", material_class, XSD.string)
    create_data_property("materialCode", material_class, XSD.string)
    
    # BridgeElement (existing)
    create_data_property("YEAR", bridge_element_class, XSD.gYear)
    create_data_property("ELEMENT_NUMBER", bridge_element_class, XSD.integer)
    create_data_property("ELEMENT_NAME", bridge_element_class, XSD.string)
    create_data_property("CONDITION_STATE_1", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_1", bridge_element_class, XSD.decimal)
    create_data_property("CONDITION_STATE_2", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_2", bridge_element_class, XSD.decimal)
    create_data_property("CONDITION_STATE_3", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_3", bridge_element_class, XSD.decimal)
    create_data_property("CONDITION_STATE_4", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_4", bridge_element_class, XSD.decimal)
    create_data_property("HI", bridge_element_class, XSD.decimal)
    
    # Location (existing)
    create_data_property("NBI_1 - State Code", location_class, XSD.string)
    create_data_property("NBI_1 - State Name", location_class, XSD.string)
    create_data_property("NBI_3 - County Name", location_class, XSD.string)
    create_data_property("NBI_16 - Latitude (decimal)", location_class, XSD.decimal)
    create_data_property("NBI_17 - Longitude (decimal)", location_class, XSD.decimal)
    
    # WeatherExposure (existing)
    create_data_property("NBI_Number of Freeze-Thaw Cycles", weather_exposure_class, XSD.integer)
    create_data_property("NBI_Number of Snowfall Days", weather_exposure_class, XSD.integer)
    create_data_property("NBI_Average Temperature", weather_exposure_class, XSD.decimal)
    create_data_property("NBI_Total Precipitation", weather_exposure_class, XSD.decimal)
    
    # TrafficProfile (existing)
    create_data_property("NBI_29 - Average Daily Traffic", traffic_profile_class, XSD.integer)
    create_data_property("NBI_109 - Average Daily Truck Traffic (Percent ADT)", traffic_profile_class, XSD.decimal)
    create_data_property("NBI_Computed - Average Daily Truck Traffic (Volume)", traffic_profile_class, XSD.integer)
    
    # Inspection (existing)
    create_data_property("NBI_90 - Inspection Date", inspection_class, XSD.date)
    create_data_property("NBI_91 - Designated Inspection Frequency", inspection_class, XSD.integer)
    
    # ConditionRating (existing)
    create_data_property("NBI_58 - Deck Condition Rating", condition_rating_class, XSD.string)
    create_data_property("NBI_59 - Superstructure Condition Rating", condition_rating_class, XSD.string)
    create_data_property("NBI_60 - Substructure Condition Rating", condition_rating_class, XSD.string)
    
    # MaintenanceAction (existing)
    create_data_property("NBI_94 - Bridge Improvement Cost", maintenance_action_class, XSD.decimal)
    create_data_property("NBI_96 - Total Project Cost", maintenance_action_class, XSD.decimal)
    
    # --- 8. CLOSE CONNECTION AND COMMIT ---
    g.close(True)
    
    print("\n--- SUCCESS! ---")
    print("The enhanced ontology schema with GNN relationships has been pushed to Neo4j.")
    print("New additions:")
    print("  - ClimateZone nodes for regional weather classification")
    print("  - TrafficPattern nodes for traffic intensity grouping")
    print("  - Material nodes for construction material types")
    print("  - NEAR relationships for spatial proximity")
    print("  - SIMILAR_TO relationships for bridge similarity")
    print("  - Computed properties: adtl, truckPercentage, criticalityScore")

except Exception as e:
    print("\n--- ERROR ---")
    print(f"An error occurred during the rdflib-neo4j process: {e}")
    print("\nPlease check:")
    print("1. Is your Neo4j database running?")
    print("2. Did you set the correct 'password' variable in this script?")
    print("3. Did you install the libraries? (!pip install neo4j rdflib-neo4j)")

Connecting directly to Neo4j to clear database...
Database cleared successfully.
Connecting to Neo4j at bolt://localhost:7687 with rdflib...
Uniqueness constraint on :Resource(uri) is created.
The store is now: Open
rdflib connection successful. Setting up namespaces...
Defining classes and writing to Neo4j...
Defining object properties and writing to Neo4j...
Defining data properties and writing to Neo4j...
The store is now: Closed
IMPORTED 334 TRIPLES

--- SUCCESS! ---
The enhanced ontology schema with GNN relationships has been pushed to Neo4j.
New additions:
  - ClimateZone nodes for regional weather classification
  - TrafficPattern nodes for traffic intensity grouping
  - Material nodes for construction material types
  - NEAR relationships for spatial proximity
  - SIMILAR_TO relationships for bridge similarity
  - Computed properties: adtl, truckPercentage, criticalityScore


# Data population

In [3]:
import pandas as pd
from neo4j import GraphDatabase
import re
import math

# --- 1. SET UP NEO4J CONNECTION DETAILS ---
uri = "bolt://localhost:7687"
user = "neo4j"
password = "MikalBridges"
database = "neo4j"

# --- 2. SET UP CSV FILE PATH ---
csv_file_path = "oklahoma/NEW_NBE_and_NBI_OK.csv"

def clean_data(data):
    """
    Cleans data for Neo4j import.
    - Converts pandas NaN/NaT to None (required for Neo4j driver).
    - Converts numpy types to Python native types.
    - LEAVES STRINGS UNTOUCHED (No stripping).
    """
    # Handle pandas missing values (NaN, NaT, None)
    if pd.isna(data):
        return None
    
    # Handle numpy types (e.g., np.int64, np.float64) -> Python types
    if hasattr(data, 'item'):
        val = data.item()
        # Re-check for NaN after conversion (sometimes happens with numpy floats)
        if pd.isna(val):
            return None
        return val
    
    # Return everything else (including strings) exactly as is
    return data

def compute_climate_zone(freeze_thaw_cycles):
    """Classify climate zone based on freeze-thaw cycles"""
    if freeze_thaw_cycles is None or pd.isna(freeze_thaw_cycles):
        return 'MILD'  # Default to MILD instead of UNKNOWN
    
    # Convert to float to handle any numeric type
    try:
        cycles = float(freeze_thaw_cycles)
    except (ValueError, TypeError):
        return 'MILD'
    
    if cycles >= 80:
        return 'SEVERE'
    elif cycles >= 40:
        return 'MODERATE'
    else:
        return 'MILD'

def compute_traffic_pattern(adtl):
    """Classify traffic pattern based on ADTL (traffic per lane)"""
    if adtl is None or pd.isna(adtl):
        return 'LOW'  # Default to LOW instead of UNKNOWN
    
    # Convert to float to handle any numeric type
    try:
        traffic = float(adtl)
    except (ValueError, TypeError):
        return 'LOW'
    
    if traffic >= 2000:
        return 'HIGH'
    elif traffic >= 500:
        return 'MEDIUM'
    else:
        return 'LOW'

# --- 4. BATCH PROCESSING FUNCTION ---
def process_batch(session, batch_data):
    """
    Enhanced batch processing with GNN-specific features and relationships
    """
    
    cypher_query = """
    UNWIND $batch AS row
    
    // --- 1. Create/Merge the core :Bridge node with COMPUTED PROPERTIES ---
    MERGE (b:Bridge {structureNumber: row.structure_number})
    ON CREATE SET
        b.latestStructureNumber = row.latest_structure_number,
        b.yearBuilt = row.year_built,
        b.bridgeAge = row.bridge_age,
        b.mainSpanMaterial = row.main_span_material,
        b.mainSpanDesign = row.main_span_design,
        b.numMainSpans = row.num_main_spans,
        b.structureLength = row.structure_length,
        b.deckWidth = row.deck_width,
        b.deckArea = row.deck_area,
        b.ownerAgency = row.owner_agency,
        b.functionalClass = row.functional_class,
        b.latitude = row.latitude,
        b.longitude = row.longitude,
        // *** NEW: Computed properties for GNN ***
        b.adtl = row.adtl,  // Traffic per lane
        b.truckPercentage = row.truck_percentage,
        b.numLanes = row.num_lanes,
        b.adt = row.adt,
        b.adtt = row.adtt_volume
    
    // Always update these with latest data
    SET b.currentDeckCR = row.deck_condition,
        b.currentSuperCR = row.superstructure_condition,
        b.currentSubCR = row.substructure_condition,
        b.lastInspectionDate = row.inspection_date,
        b.healthIndex = row.hi
    
    // --- 2. Create/Merge :Material node and link ---
    WITH b, row
    WHERE row.main_span_material IS NOT NULL
    MERGE (m:Material {materialType: row.main_span_material})
    ON CREATE SET m.materialCode = row.main_span_material_code
    MERGE (b)-[:CONSTRUCTED_WITH]->(m)
    
    // --- 3. Create/Merge :Location node and link ---
    WITH b, row
    WHERE row.state_name IS NOT NULL AND row.county_name IS NOT NULL
    MERGE (l:Location {name: row.county_name + ', ' + row.state_name})
    ON CREATE SET
        l.stateCode = row.state_code,
        l.stateName = row.state_name,
        l.countyCode = row.county_code,
        l.countyName = row.county_name,
        l.latitude = row.latitude,
        l.longitude = row.longitude
    MERGE (b)-[:LOCATED_IN]->(l)
    
    // --- 4. Create/Merge :ClimateZone node and link (NEW FOR GNN) ---
    WITH b, l, row
    WHERE row.climate_zone IS NOT NULL
    MERGE (cz:ClimateZone {zoneID: row.climate_zone})
    ON CREATE SET
        cz.severity = row.climate_zone,
        cz.description = CASE row.climate_zone
            WHEN 'SEVERE' THEN 'High freeze-thaw exposure (80+ cycles/year)'
            WHEN 'MODERATE' THEN 'Moderate freeze-thaw exposure (40-80 cycles/year)'
            WHEN 'MILD' THEN 'Low freeze-thaw exposure (<40 cycles/year)'
            ELSE 'Unknown climate exposure'
        END
    // Create relationship WITHOUT properties first, then SET non-null values
    MERGE (b)-[e:EXPOSED_TO]->(cz)
    SET e.freezeThawCycles = CASE WHEN row.freeze_thaw_cycles IS NOT NULL THEN row.freeze_thaw_cycles ELSE e.freezeThawCycles END,
        e.snowfallDays = CASE WHEN row.snowfall_days IS NOT NULL THEN row.snowfall_days ELSE e.snowfallDays END,
        e.avgTemp = CASE WHEN row.avg_temp IS NOT NULL THEN row.avg_temp ELSE e.avgTemp END,
        e.totalPrecip = CASE WHEN row.total_precipitation IS NOT NULL THEN row.total_precipitation ELSE e.totalPrecip END
    
    // --- 5. Create/Merge :WeatherExposure node and link to Location ---
    WITH b, l, cz, row
    WHERE row.state_name IS NOT NULL AND row.county_name IS NOT NULL
    MERGE (w:WeatherExposure {weatherID: row.county_name + ', ' + row.state_name})
    ON CREATE SET
        w.avgRelativeHumidity = row.avg_humidity,
        w.avgTemp = row.avg_temp,
        w.maxTemp = row.max_temp,
        w.minTemp = row.min_temp,
        w.freezeThawCycles = row.freeze_thaw_cycles,
        w.snowfallDays = row.snowfall_days,
        w.precipitationDays = row.precipitation_days,
        w.totalPrecipitation = row.total_precipitation
    MERGE (l)-[:HAS_EXPOSURE]->(w)
    
    // --- 6. Create/Merge :TrafficPattern node and link (NEW FOR GNN) ---
    WITH b, row
    WHERE row.traffic_pattern IS NOT NULL
    MERGE (tp:TrafficPattern {patternID: row.traffic_pattern})
    ON CREATE SET
        tp.intensity = row.traffic_pattern,
        tp.description = CASE row.traffic_pattern
            WHEN 'HIGH' THEN 'High traffic volume (2000+ vehicles/lane/day)'
            WHEN 'MEDIUM' THEN 'Medium traffic volume (500-2000 vehicles/lane/day)'
            WHEN 'LOW' THEN 'Low traffic volume (<500 vehicles/lane/day)'
            ELSE 'Unknown traffic volume'
        END
    // Create relationship WITHOUT properties first, then SET non-null values
    MERGE (b)-[t:HAS_TRAFFIC]->(tp)
    SET t.adt = CASE WHEN row.adt IS NOT NULL THEN row.adt ELSE t.adt END,
        t.adtt = CASE WHEN row.adtt_volume IS NOT NULL THEN row.adtt_volume ELSE t.adtt END,
        t.adtl = CASE WHEN row.adtl IS NOT NULL THEN row.adtl ELSE t.adtl END,
        t.truckPercentage = CASE WHEN row.truck_percentage IS NOT NULL THEN row.truck_percentage ELSE t.truckPercentage END,
        t.yearOfAdt = CASE WHEN row.year_of_adt IS NOT NULL THEN row.year_of_adt ELSE t.yearOfAdt END
    
    // --- 7. Create/Merge :TrafficProfile node (original) ---
    WITH b, row
    WHERE row.year_of_adt IS NOT NULL AND row.structure_number IS NOT NULL
    MERGE (tf:TrafficProfile {trafficID: row.structure_number + '_' + row.year_of_adt})
    ON CREATE SET
        tf.adt = row.adt,
        tf.adttPercent = row.adtt_percent,
        tf.yearOfAdt = row.year_of_adt,
        tf.adttVolume = row.adtt_volume
    MERGE (b)-[:HAS_TRAFFIC_PROFILE]->(tf)
    
    // --- 8. Create/Merge :Inspection node ---
    WITH b, row
    WHERE row.inspection_date IS NOT NULL AND row.structure_number IS NOT NULL
    MERGE (i:Inspection:Event:MaintenanceActivity {inspectionID: row.structure_number + '_' + row.inspection_date})
    ON CREATE SET
        i.nbiYear = row.nbi_year,
        i.inspectionDate = row.inspection_date,
        i.inspFreq = row.insp_freq
    MERGE (i)-[:IS_PERFORMED_ON]->(b)
    
    // --- 9. Create/Merge :ConditionRating node ---
    WITH b, i, row
    WHERE i.inspectionID IS NOT NULL
    MERGE (cr:ConditionRating {conditionRatingID: i.inspectionID})
    ON CREATE SET
        cr.deckCondition = row.deck_condition,
        cr.superstructureCondition = row.superstructure_condition,
        cr.substructureCondition = row.substructure_condition,
        cr.bridgeCondition = row.bridge_condition
    MERGE (i)-[:RESULTED_IN]->(cr)
    
    // --- 10. Create/Merge :BridgeElement node ---
    WITH b, i, cr, row
    WHERE row.element_name IS NOT NULL 
        AND row.structure_number IS NOT NULL 
        AND row.element_number IS NOT NULL 
        AND row.year IS NOT NULL
    
    MERGE (be:BridgeElement:Asset {elementID: row.structure_number + '_' + row.element_number + '_' + row.year})
    ON CREATE SET
        be.year = row.year,
        be.elementNumber = row.element_number,
        be.elementCategory = row.element_category,
        be.elementName = row.element_name,
        be.unit = row.element_unit,
        be.totalQuantity = row.total_quantity,
        be.cs1 = row.cs1,
        be.cs1_percent = row.cs1_percent,
        be.cs2 = row.cs2,
        be.cs2_percent = row.cs2_percent,
        be.cs3 = row.cs3,
        be.cs3_percent = row.cs3_percent,
        be.cs4 = row.cs4,
        be.cs4_percent = row.cs4_percent,
        be.hi = row.hi
    
    MERGE (b)-[:HAS_ELEMENT]->(be)
    MERGE (be)-[:INSPECTED_DURING]->(i)
    
    // --- 11. Create/Merge :MaintenanceAction node ---
    WITH b, row
    WHERE (row.improvement_cost IS NOT NULL OR row.total_project_cost IS NOT NULL)
        AND row.structure_number IS NOT NULL 
        AND row.year_of_cost IS NOT NULL
    
    MERGE (ma:MaintenanceAction:Event:MaintenanceActivity {actionID: row.structure_number + '_' + row.year_of_cost})
    ON CREATE SET
        ma.bridgeImprovementCost = row.improvement_cost,
        ma.totalProjectCost = row.total_project_cost,
        ma.yearOfCostEstimate = row.year_of_cost
    MERGE (ma)-[:IS_PERFORMED_ON]->(b)
    """
    
    session.run(cypher_query, batch=batch_data)

def create_spatial_relationships(session, distance_km=50):
    """
    Create NEAR relationships between spatially proximate bridges
    This is crucial for GNN message passing
    """
    print(f"\nCreating spatial NEAR relationships (within {distance_km} km)...")
    
    query = """
    MATCH (b1:Bridge), (b2:Bridge)
    WHERE b1.structureNumber < b2.structureNumber
        AND b1.latitude IS NOT NULL 
        AND b1.longitude IS NOT NULL
        AND b2.latitude IS NOT NULL 
        AND b2.longitude IS NOT NULL
    WITH b1, b2,
         point.distance(
             point({latitude: b1.latitude, longitude: b1.longitude}),
             point({latitude: b2.latitude, longitude: b2.longitude})
         ) / 1000.0 AS distance_km
    WHERE distance_km < $threshold
    MERGE (b1)-[r:NEAR]->(b2)
    ON CREATE SET r.distanceKm = distance_km
    MERGE (b2)-[r2:NEAR]->(b1)
    ON CREATE SET r2.distanceKm = distance_km
    RETURN count(r) as relationships_created
    """
    
    result = session.run(query, threshold=distance_km)
    count = result.single()['relationships_created']
    print(f"  ✓ Created {count} NEAR relationships")

def create_similarity_relationships(session):
    """
    Create SIMILAR_TO relationships based on bridge characteristics
    Bridges are similar if they share: material, age range, traffic pattern
    """
    print("\nCreating SIMILAR_TO relationships...")
    
    query = """
    MATCH (b1:Bridge)-[:CONSTRUCTED_WITH]->(m:Material)<-[:CONSTRUCTED_WITH]-(b2:Bridge)
    WHERE b1.structureNumber < b2.structureNumber
        AND abs(b1.bridgeAge - b2.bridgeAge) < 10
        AND b1.functionalClass = b2.functionalClass
    WITH b1, b2,
         abs(b1.adt - b2.adt) * 1.0 / CASE WHEN b1.adt > b2.adt THEN b1.adt ELSE b2.adt END AS traffic_diff
    WHERE traffic_diff < 0.5
    MERGE (b1)-[r:SIMILAR_TO]->(b2)
    ON CREATE SET r.similarityScore = 1.0 - traffic_diff
    MERGE (b2)-[r2:SIMILAR_TO]->(b1)
    ON CREATE SET r2.similarityScore = 1.0 - traffic_diff
    RETURN count(r) as relationships_created
    """
    
    result = session.run(query)
    count = result.single()['relationships_created']
    print(f"  ✓ Created {count} SIMILAR_TO relationships")

def create_indexes_and_constraints(session):
    """Create additional indexes for GNN queries"""
    print("\nCreating indexes for GNN performance...")
    
    indexes = [
        "CREATE INDEX bridge_state IF NOT EXISTS FOR (b:Bridge) ON (b.functionalClass)",
        "CREATE INDEX bridge_material IF NOT EXISTS FOR (b:Bridge) ON (b.mainSpanMaterial)",
        "CREATE INDEX bridge_age IF NOT EXISTS FOR (b:Bridge) ON (b.bridgeAge)",
        "CREATE INDEX bridge_cr IF NOT EXISTS FOR (b:Bridge) ON (b.currentDeckCR)",
        "CREATE INDEX location_state IF NOT EXISTS FOR (l:Location) ON (l.stateName)",
        "CREATE INDEX climate_zone IF NOT EXISTS FOR (cz:ClimateZone) ON (cz.zoneID)",
        "CREATE INDEX traffic_pattern IF NOT EXISTS FOR (tp:TrafficPattern) ON (tp.patternID)"
    ]
    
    for index in indexes:
        try:
            session.run(index)
        except Exception as e:
            # Index might already exist
            pass
    
    print("  ✓ Indexes created")

# --- 5. MAIN EXECUTION SCRIPT ---
def main():
    driver = None
    try:
        # --- Connect to Neo4j ---
        driver = GraphDatabase.driver(uri, auth=(user, password))
        with driver.session(database=database) as session:
            print("Successfully connected to Neo4j.")
            
            # --- Create Constraints ---
            print("Setting up database constraints...")
            
            constraints = [
                "CREATE CONSTRAINT IF NOT EXISTS FOR (b:Bridge) REQUIRE b.structureNumber IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.name IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (w:WeatherExposure) REQUIRE w.weatherID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (t:TrafficProfile) REQUIRE t.trafficID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (i:Inspection) REQUIRE i.inspectionID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (ma:MaintenanceAction) REQUIRE ma.actionID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (be:BridgeElement) REQUIRE be.elementID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (cr:ConditionRating) REQUIRE cr.conditionRatingID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (cz:ClimateZone) REQUIRE cz.zoneID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (tp:TrafficPattern) REQUIRE tp.patternID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (m:Material) REQUIRE m.materialType IS UNIQUE"
            ]

            for constraint in constraints:
                session.run(constraint)
            print("Constraints are set.")

            # --- Load data from CSV ---
            print(f"\nLoading data from '{csv_file_path}'...")
            try:
                df = pd.read_csv(csv_file_path, low_memory=False)
            except FileNotFoundError:
                print(f"--- FATAL ERROR ---")
                print(f"File not found: {csv_file_path}")
                return
            
            print(f"Loaded {len(df)} rows from CSV.")
            
            # --- COMPUTE DERIVED FEATURES FOR GNN ---
            print("\nComputing GNN features...")
            
            # Get the most recent inspection data per bridge
            df = df.sort_values(['STRUCTURE_NUMBER', 'NBI_90 - Inspection Date'], ascending=[True, False])
            
            # Compute ADTL (traffic per lane) - handle missing lanes
            df['num_lanes'] = df.get('NBI_28A - Lanes On Structure', 2)
            df['num_lanes'] = df['num_lanes'].fillna(2)  # Default to 2 lanes
            df['num_lanes'] = df['num_lanes'].replace(0, 2)  # Replace 0 with 2
            
            # Compute ADTL safely
            df['adtl'] = df['NBI_29 - Average Daily Traffic'] / df['num_lanes']
            df['adtl'] = df['adtl'].fillna(0)  # Fill NaN with 0
            
            # Compute truck percentage safely
            adt_safe = df['NBI_29 - Average Daily Traffic'].fillna(1)  # Avoid division by zero
            adt_safe = adt_safe.replace(0, 1)
            df['truck_percentage'] = (
                df['NBI_Computed - Average Daily Truck Traffic (Volume)'].fillna(0) / 
                adt_safe * 100
            )
            df['truck_percentage'] = df['truck_percentage'].fillna(0)
            
            # Classify climate zone - handle NaN
            df['NBI_Number of Freeze-Thaw Cycles'] = df['NBI_Number of Freeze-Thaw Cycles'].fillna(0)
            df['climate_zone'] = df['NBI_Number of Freeze-Thaw Cycles'].apply(compute_climate_zone)
            
            # Classify traffic pattern
            df['traffic_pattern'] = df['adtl'].apply(compute_traffic_pattern)
            
            # Get material code
            df['main_span_material_code'] = df['NBI_43A - Main Span Material']
            
            print(f"  ✓ Computed adtl, truck_percentage, climate_zone, traffic_pattern")
            
            # --- Prepare data for batching ---
            column_rename_map = {
                'YEAR': 'year',
                'STRUCTURE_NUMBER': 'structure_number',
                'LATEST_STRUCTURE_NUMBER': 'latest_structure_number',
                'OWNER_AGENCY': 'owner_agency',
                'ELEMENT_CATEGORY': 'element_category',
                'PARENT_ELEMENT_NUMBER': 'parent_element_number',
                'ELEMENT_NUMBER': 'element_number',
                'ELEMENT_NAME': 'element_name',
                'ELEMENT_UNIT': 'element_unit',
                'TOTAL_QUANTITY': 'total_quantity',
                'CONDITION_STATE_1': 'cs1',
                'CS_PERCENT_1': 'cs1_percent',
                'CONDITION_STATE_2': 'cs2',
                'CS_PERCENT_2': 'cs2_percent',
                'CONDITION_STATE_3': 'cs3',
                'CS_PERCENT_3': 'cs3_percent',
                'CONDITION_STATE_4': 'cs4',
                'CS_PERCENT_4': 'cs4_percent',
                'HI': 'hi',
                'NBI_Year': 'nbi_year',
                'NBI_1 - State Code': 'state_code',
                'NBI_1 - State Name': 'state_name',
                'NBI_3 - County Code': 'county_code',
                'NBI_3 - County Name': 'county_name',
                'NBI_27 - Year Built': 'year_built',
                'NBI_29 - Average Daily Traffic': 'adt',
                'NBI_43A - Main Span Material': 'main_span_material',
                'NBI_43B - Main Span Design': 'main_span_design',
                'NBI_45 - Number of Spans in Main Unit': 'num_main_spans',
                'NBI_49 - Structure Length (ft.)': 'structure_length',
                'NBI_Bridge Age (yr)': 'bridge_age',
                'NBI_CAT29 - Deck Area (sq. ft.)': 'deck_area',
                'NBI_16 - Latitude (decimal)': 'latitude',
                'NBI_17 - Longitude (decimal)': 'longitude',
                'NBI_26 - Functional Classification of Inventory Route': 'functional_class',
                'NBI_52 - Deck Width - Out to Out (ft.)': 'deck_width',
                'NBI_90 - Inspection Date': 'inspection_date',
                'NBI_91 - Designated Inspection Frequency': 'insp_freq',
                'NBI_58 - Deck Condition Rating': 'deck_condition',
                'NBI_59 - Superstructure Condition Rating': 'superstructure_condition',
                'NBI_60 - Substructure Condition Rating': 'substructure_condition',
                'NBI_CAT10 - Bridge Condition': 'bridge_condition',
                'NBI_30 - Year of Average Daily Traffic': 'year_of_adt',
                'NBI_109 - Average Daily Truck Traffic (Percent ADT)': 'adtt_percent',
                'NBI_Computed - Average Daily Truck Traffic (Volume)': 'adtt_volume',
                'NBI_94 - Bridge Improvement Cost': 'improvement_cost',
                'NBI_96 - Total Project Cost': 'total_project_cost',
                'NBI_97 - Year of Improvement Cost Estimate': 'year_of_cost',
                'NBI_Number of Freeze-Thaw Cycles': 'freeze_thaw_cycles',
                'NBI_Number of Snowfall Days': 'snowfall_days',
                'NBI_Number of Days with Measurable Precipitation': 'precipitation_days',
                'NBI_Average Temperature': 'avg_temp',
                'NBI_Average Relative Humidity': 'avg_humidity',
                'NBI_Maximum Temperature': 'max_temp',
                'NBI_Minimum Temperature': 'min_temp',
                'NBI_Total Precipitation': 'total_precipitation',
                # New computed columns
                'adtl': 'adtl',
                'truck_percentage': 'truck_percentage',
                'climate_zone': 'climate_zone',
                'traffic_pattern': 'traffic_pattern',
                'main_span_material_code': 'main_span_material_code',
                'num_lanes': 'num_lanes'
            }
            
            original_cols_to_keep = [col for col in column_rename_map.keys() if col in df.columns]
            data_df = df[original_cols_to_keep].rename(columns=column_rename_map)

            # Clean data
            data_for_neo4j = [
                {key: clean_data(value) for key, value in row.items()}
                for row in data_df.to_dict('records')
            ]
            
            # --- Process in batches ---
            batch_size = 500
            total_rows = len(data_for_neo4j)
            
            print(f"\nStarting to import {total_rows} rows in batches of {batch_size}...")
            
            for i in range(0, total_rows, batch_size):
                batch = data_for_neo4j[i:min(i + batch_size, total_rows)]
                process_batch(session, batch)
                if (i // batch_size) % 10 == 0:  # Print every 10 batches
                    print(f"  ... processed {min(i + batch_size, total_rows)}/{total_rows} rows")
            
            print(f"\n✓ Data import complete!")
            
            # --- Create GNN-specific relationships ---
            create_spatial_relationships(session, distance_km=50)
            create_similarity_relationships(session)
            
            # --- Create indexes ---
            create_indexes_and_constraints(session)
            
            # --- Print summary statistics ---
            print("\n" + "="*60)
            print("DATABASE SUMMARY")
            print("="*60)
            
            stats_query = """
            MATCH (b:Bridge) 
            OPTIONAL MATCH (b)-[:NEAR]-(n)
            WITH b, count(DISTINCT n) as near_count
            RETURN 
                count(b) as total_bridges,
                avg(near_count) as avg_neighbors,
                max(near_count) as max_neighbors
            """
            stats = session.run(stats_query).single()
            
            print(f"Total Bridges: {stats['total_bridges']}")
            print(f"Avg NEAR neighbors: {stats['avg_neighbors']:.1f}")
            print(f"Max NEAR neighbors: {stats['max_neighbors']}")
            
            node_counts = """
            MATCH (n)
            RETURN labels(n)[0] as label, count(n) as count
            ORDER BY count DESC
            """
            print("\nNode Counts:")
            for record in session.run(node_counts):
                print(f"  {record['label']}: {record['count']}")
            
            print("\n" + "="*60)
            print("✓ SUCCESS! Your graph is ready for GNN training")
            print("="*60)
            print("\nNext steps:")
            print("1. Run the feature extraction script to create PyTorch Geometric data")
            print("2. Train the GNN model")
            print("3. Generate criticality scores")

    except Exception as e:
        print("\n--- FATAL ERROR ---")
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        if driver:
            driver.close()
            print("\nNeo4j connection closed.")

# --- RUN THE SCRIPT ---
if __name__ == "__main__":
    main()

Successfully connected to Neo4j.
Setting up database constraints...
Constraints are set.

Loading data from 'oklahoma/NEW_NBE_and_NBI_OK.csv'...
Loaded 101243 rows from CSV.

Computing GNN features...
  ✓ Computed adtl, truck_percentage, climate_zone, traffic_pattern

Starting to import 101243 rows in batches of 500...
  ... processed 500/101243 rows
  ... processed 5500/101243 rows
  ... processed 10500/101243 rows
  ... processed 15500/101243 rows
  ... processed 20500/101243 rows
  ... processed 25500/101243 rows
  ... processed 30500/101243 rows
  ... processed 35500/101243 rows
  ... processed 40500/101243 rows
  ... processed 45500/101243 rows
  ... processed 50500/101243 rows
  ... processed 55500/101243 rows
  ... processed 60500/101243 rows
  ... processed 65500/101243 rows
  ... processed 70500/101243 rows
  ... processed 75500/101243 rows
  ... processed 80500/101243 rows
  ... processed 85500/101243 rows
  ... processed 90500/101243 rows
  ... processed 95500/101243 rows
  

## Extract Neo4j Graph Data for PyTorch Geometric GNN


In [1]:

import torch
from torch_geometric.data import Data
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- NEO4J CONNECTION ---
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "MikalBridges"  

print("="*60)
print("EXTRACTING GRAPH DATA FROM NEO4J")
print("="*60)

# --- 1. CONNECT TO NEO4J ---
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def extract_bridge_features():
    """Extract bridge node features for GNN"""
    
    query = """
    MATCH (b:Bridge)
    OPTIONAL MATCH (b)-[e:EXPOSED_TO]->(cz:ClimateZone)
    OPTIONAL MATCH (b)-[t:HAS_TRAFFIC]->(tp:TrafficPattern)
    RETURN 
        b.structureNumber as bridge_id,
        b.bridgeAge as age,
        b.adtl as adtl,
        b.truckPercentage as truck_pct,
        b.currentDeckCR as deck_cr,
        b.currentSuperCR as super_cr,
        b.currentSubCR as sub_cr,
        b.healthIndex as health_index,
        e.freezeThawCycles as freeze_thaw,
        e.snowfallDays as snowfall,
        b.structureLength as length,
        b.deckArea as deck_area,
        b.numLanes as num_lanes,
        b.functionalClass as functional_class,
        CASE b.mainSpanMaterial
            WHEN 'Steel' THEN 1
            WHEN 'Concrete' THEN 2
            WHEN 'Prestressed Concrete' THEN 3
            WHEN 'Timber' THEN 4
            ELSE 0
        END as material_code,
        CASE cz.zoneID
            WHEN 'SEVERE' THEN 2
            WHEN 'MODERATE' THEN 1
            WHEN 'MILD' THEN 0
            ELSE 0
        END as climate_code,
        CASE tp.patternID
            WHEN 'HIGH' THEN 2
            WHEN 'MEDIUM' THEN 1
            WHEN 'LOW' THEN 0
            ELSE 0
        END as traffic_code
    ORDER BY bridge_id
    """
    
    with driver.session() as session:
        result = session.run(query)
        df = pd.DataFrame([dict(record) for record in result])
    
    print(f"\n✓ Extracted {len(df)} bridge nodes")
    
    # Show data quality
    print("\nData Quality Check:")
    print(f"  Missing age: {df['age'].isna().sum()}")
    print(f"  Missing deck_cr: {df['deck_cr'].isna().sum()}")
    print(f"  Missing adtl: {df['adtl'].isna().sum()}")
    print(f"  Missing freeze_thaw: {df['freeze_thaw'].isna().sum()}")
    
    return df

def extract_edge_indices(bridge_ids):
    """Extract NEAR relationship edges"""
    
    query = """
    MATCH (b1:Bridge)-[r:NEAR]->(b2:Bridge)
    RETURN b1.structureNumber as source, b2.structureNumber as target
    """
    
    with driver.session() as session:
        result = session.run(query)
        edges = [(record['source'], record['target']) for record in result]
    
    print(f"\n✓ Extracted {len(edges)} NEAR edges")
    
    # Create bridge ID to index mapping
    id_to_idx = {bridge_id: idx for idx, bridge_id in enumerate(bridge_ids)}
    
    # Convert to PyTorch format
    edge_index = []
    for src, tgt in edges:
        if src in id_to_idx and tgt in id_to_idx:
            edge_index.append([id_to_idx[src], id_to_idx[tgt]])
    
    if len(edge_index) == 0:
        print("\n⚠️  WARNING: No valid edges found!")
        print("   This might happen if bridges lack NEAR relationships.")
        print("   The GNN will still work but without spatial message passing.")
        # Create self-loops as fallback
        edge_index = [[i, i] for i in range(len(bridge_ids))]
    
    edge_tensor = torch.LongTensor(edge_index).t().contiguous()
    
    return edge_tensor

def create_pyg_data():
    """Create PyTorch Geometric Data object"""
    
    print("\n" + "="*60)
    print("STEP 1: Extracting Bridge Features")
    print("="*60)
    
    # Extract features
    df = extract_bridge_features()
    bridge_ids = df['bridge_id'].values
    
    # Define feature columns
    feature_cols = [
        'age', 'adtl', 'truck_pct', 'deck_cr', 'super_cr', 'sub_cr',
        'health_index', 'freeze_thaw', 'snowfall', 'length', 'deck_area',
        'num_lanes', 'material_code', 'climate_code', 'traffic_code'
    ]
    
    # Handle missing values
    print("\nFilling missing values with column means...")
    for col in feature_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mean())
        else:
            print(f"⚠️  Column '{col}' not found, setting to 0")
            df[col] = 0
    
    # Create feature matrix
    X = df[feature_cols].values.astype(np.float32)
    
    # Standardize features (important for GNN training!)
    print("Standardizing features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_tensor = torch.FloatTensor(X_scaled)
    
    print(f"\n✓ Created feature matrix: {X_tensor.shape}")
    print(f"  Features per bridge: {X_tensor.shape[1]}")
    
    print("\n" + "="*60)
    print("STEP 2: Extracting Graph Edges")
    print("="*60)
    
    # Extract edges
    edge_index = extract_edge_indices(bridge_ids)
    
    print(f"\n✓ Created edge index: {edge_index.shape}")
    print(f"  Edges: {edge_index.shape[1]}")
    
    print("\n" + "="*60)
    print("STEP 3: Creating Labels (for training)")
    print("="*60)
    
    # Create labels based on deck condition rating
    # We'll predict if a bridge needs attention (deck CR <= 5)
    labels = (df['deck_cr'] <= 5).astype(int).values
    y = torch.LongTensor(labels)
    
    num_critical = (labels == 1).sum()
    num_good = (labels == 0).sum()
    
    print(f"\n✓ Created labels:")
    print(f"  Critical bridges (CR ≤ 5): {num_critical} ({num_critical/len(labels)*100:.1f}%)")
    print(f"  Good bridges (CR > 5): {num_good} ({num_good/len(labels)*100:.1f}%)")
    
    # Create train/val/test masks
    n = len(bridge_ids)
    indices = torch.randperm(n)
    
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    
    train_mask = torch.zeros(n, dtype=torch.bool)
    val_mask = torch.zeros(n, dtype=torch.bool)
    test_mask = torch.zeros(n, dtype=torch.bool)
    
    train_mask[indices[:n_train]] = True
    val_mask[indices[n_train:n_train+n_val]] = True
    test_mask[indices[n_train+n_val:]] = True
    
    print(f"\n✓ Created train/val/test split:")
    print(f"  Training: {train_mask.sum()} bridges")
    print(f"  Validation: {val_mask.sum()} bridges")
    print(f"  Testing: {test_mask.sum()} bridges")
    
    # Create PyG Data object
    data = Data(
        x=X_tensor,
        edge_index=edge_index,
        y=y,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask
    )
    
    # Store metadata
    data.bridge_ids = bridge_ids
    data.feature_names = feature_cols
    data.num_classes = 2  # Binary: critical (1) or good (0)
    
    return data, scaler

# --- RUN EXTRACTION ---
print("\nStarting data extraction...\n")

try:
    data, scaler = create_pyg_data()
    
    # Save the data
    torch.save(data, 'oklahoma_bridge_graph.pt')
    torch.save(scaler, 'feature_scaler.pt')
    
    print("\n" + "="*60)
    print("✅ SUCCESS! DATA READY FOR GNN TRAINING")
    print("="*60)
    print("\nSaved files:")
    print("  - oklahoma_bridge_graph.pt (graph data)")
    print("  - feature_scaler.pt (for inference)")
    
    print("\nGraph Statistics:")
    print(f"  Nodes: {data.num_nodes}")
    print(f"  Edges: {data.num_edges}")
    print(f"  Features per node: {data.num_node_features}")
    print(f"  Average degree: {data.num_edges / data.num_nodes:.2f}")
    
    print("\n" + "="*60)
    print("NEXT STEP: Run the GNN training script!")
    print("="*60)

except Exception as e:
    print(f"\n❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.close()
    print("\nNeo4j connection closed.")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\aksel\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.